# Day 2: Assignment — Production-Ready Extraction Pipeline

## Overview

Build and document a **production-ready extraction pipeline** using Structured Outputs.

## Deliverables

Submit ONE notebook (or PDF export) containing:

1. **Final Pydantic schema** with field descriptions
2. **Final extraction prompt** (v2 or v3)
3. **Extracted outputs** for at least 12 items
4. **Golden set** (8+ items) with ground truth labels
5. **Metrics** (accuracy for classification and urgency fields)
6. **Error analysis** (½–1 page)
7. **Prompt playbook** documentation

## Grading Criteria

| Criterion | Weight | Description |
|-----------|--------|-------------|
| Schema Design | 15% | Appropriate fields, types, constraints |
| Prompt Quality | 25% | Effective rules, examples, structure |
| Accuracy | 20% | Performance on golden set |
| Analysis | 25% | Error patterns, iteration insights |
| Documentation | 15% | Complete playbook, clear explanations |

---

## Setup

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os
import time
import json
import pandas as pd
from datetime import datetime, timezone
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from google import genai

# Configure API — reads the GEMINI_API_KEY you set up on Day 1
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except:
    API_KEY = None

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-2.5-flash-lite"

print(f"✓ API key loaded")
print(f"✓ Model: {MODEL_ID}")

In [ ]:
# Infrastructure
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate_structured(prompt, schema_model, temperature=0.2, log=True, label=None):
    """Generate structured output using Pydantic schema."""
    start_time = time.time()
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_json_schema": schema_model.model_json_schema(),
        },
    )
    
    latency = time.time() - start_time
    raw_text = response.text or ""
    result = schema_model.model_validate_json(raw_text)
    
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(),
            "label": label,
            "schema": schema_model.__name__,
            "prompt_length": len(prompt),
            "response_length": len(raw_text),
            "latency_s": round(latency, 3)
        })
    
    return result

def evaluate(predictions, golden, field):
    """Calculate accuracy for a field."""
    correct, total, errors = 0, 0, []
    for id, expected in golden.items():
        if id in predictions:
            total += 1
            pred_val = getattr(predictions[id], field)
            if pred_val == expected[field]:
                correct += 1
            else:
                errors.append({"id": id, "expected": expected[field], "got": pred_val})
    return {"correct": correct, "total": total, 
            "accuracy": correct/total if total else 0, "errors": errors}

print("✓ Infrastructure ready")

---

## Part 1: Final Schema (15 points)

Paste your final, refined Pydantic schema below.

In [ ]:
# TODO: Paste your final schema here

class FinalExtraction(BaseModel):
    """Your schema description here."""
    id: str = Field(description="Unique identifier")
    
    # TODO: Add your fields
    # category: Literal["...", "...", "..."] = Field(description="...")
    # urgency: Literal["low", "medium", "high"] = Field(description="...")
    # summary: str = Field(description="...")
    # next_step: str = Field(description="...")
    # missing_info: List[str] = Field(description="...")
    pass


class FinalBatch(BaseModel):
    """Batch of extractions."""
    items: List[FinalExtraction]


# Show schema
print("Schema fields:")
for name, field in FinalExtraction.model_fields.items():
    print(f"  {name}: {field.annotation}")

---

## Part 2: Final Prompt (25 points)

Paste your final, refined extraction prompt.

In [ ]:
# TODO: Paste your final prompt here

FINAL_PROMPT = """
# YOUR FINAL EXTRACTION PROMPT

# Include:
# - Role/context
# - Clear instructions
# - Category definitions/rules
# - Urgency definitions/rules
# - Any few-shot examples
# - Constraints

Items:
{items}
"""

def format_items(items):
    return "\n".join([f"{it['id']}: {it['text']}" for it in items])

def run_extraction(items, label="extraction"):
    prompt = FINAL_PROMPT.format(items=format_items(items))
    return generate_structured(prompt, FinalBatch, label=label)

---

## Part 3: Input Data (12+ items)

In [ ]:
# TODO: Provide your input data (at least 12 items)

inputs = [
    # {"id": "X1", "text": "..."},
    # {"id": "X2", "text": "..."},
    # ... at least 12 items
]

print(f"Input items: {len(inputs)}")
assert len(inputs) >= 12, "Need at least 12 input items!"

---

## Part 4: Run Extraction

In [ ]:
# Run extraction
# result = run_extraction(inputs, label="final_extraction")
# print(f"Extracted: {len(result.items)} items")

# View results
# df = pd.DataFrame([item.model_dump() for item in result.items])
# df

In [ ]:
# Save extracted results
# with open("day2_assignment_extracted.json", "w") as f:
#     json.dump([item.model_dump() for item in result.items], f, indent=2, ensure_ascii=False)
# print("✓ Saved: day2_assignment_extracted.json")

---

## Part 5: Golden Set (8+ items)

In [ ]:
# TODO: Define your golden set (at least 8 items with ground truth)

GOLDEN = {
    # "X1": {"category": "...", "urgency": "..."},
    # "X2": {"category": "...", "urgency": "..."},
    # ... at least 8 items
}

print(f"Golden set size: {len(GOLDEN)}")
assert len(GOLDEN) >= 8, "Need at least 8 labeled items!"

---

## Part 6: Compute Metrics (20 points)

In [ ]:
# Create predictions dict
# pred = {item.id: item for item in result.items}

# Evaluate - update field names to match your schema!
# cat_eval = evaluate(pred, GOLDEN, "category")
# urg_eval = evaluate(pred, GOLDEN, "urgency")

# print("="*50)
# print("FINAL METRICS")
# print("="*50)
# print(f"Category Accuracy: {cat_eval['correct']}/{cat_eval['total']} = {cat_eval['accuracy']:.1%}")
# print(f"Urgency Accuracy:  {urg_eval['correct']}/{urg_eval['total']} = {urg_eval['accuracy']:.1%}")

# Show errors
# if cat_eval['errors']:
#     print("\nCategory Errors:")
#     for err in cat_eval['errors']:
#         print(f"  {err['id']}: expected '{err['expected']}', got '{err['got']}'")

# if urg_eval['errors']:
#     print("\nUrgency Errors:")
#     for err in urg_eval['errors']:
#         print(f"  {err['id']}: expected '{err['expected']}', got '{err['got']}'")

---

## Part 7: Error Analysis (25 points)

Write ½–1 page covering:

### Error Analysis

#### 1. Three Common Error Patterns

**Pattern 1: [Name]**
- Description: [What happened]
- Example: [Specific case]
- Why it happens: [Root cause]

**Pattern 2: [Name]**
- Description: [What happened]
- Example: [Specific case]
- Why it happens: [Root cause]

**Pattern 3: [Name]**
- Description: [What happened]
- Example: [Specific case]
- Why it happens: [Root cause]

#### 2. Two Prompt Changes That Helped

**Change 1: [Description]**
- What I changed: [Specific modification]
- Impact: [How accuracy improved]

**Change 2: [Description]**
- What I changed: [Specific modification]
- Impact: [How accuracy improved]

#### 3. Remaining Risk + Mitigation

**Risk:** [Describe a remaining edge case or failure mode]

**Mitigation strategy:** [How you would handle this in production]
- Example: "Require human review when confidence is low" or "Add fallback rules for ambiguous cases"

---

## Part 8: Prompt Playbook (15 points)

---

# 📘 PROMPT PLAYBOOK

## Extraction Pipeline: [YOUR PIPELINE NAME]

**Version:** [e.g., 2.0]  
**Author:** [Your name]  
**Date:** [Date]  
**Status:** Production Ready / Testing

---

### Purpose

[What does this pipeline do? What business problem does it solve?]

---

### Schema

```python
# Paste your final schema here
class FinalExtraction(BaseModel):
    ...
```

| Field | Type | Description |
|-------|------|-------------|
| id | str | ... |
| category | Literal[...] | ... |
| ... | ... | ... |

---

### The Prompt

```
[Paste your final prompt here]
```

---

### Recommended Settings

| Setting | Value | Rationale |
|---------|-------|----------|
| Model | gemini-2.5-flash-lite | [Why] |
| Temperature | 0.2 | [Why - lower for structured output] |
| Structured Output | Yes | Guarantees valid JSON |

---

### Performance Metrics

| Metric | Value |
|--------|-------|
| Category Accuracy | [X]% |
| Urgency Accuracy | [X]% |
| Golden Set Size | [X] items |
| Average Latency | [X] seconds |

---

### Category Decision Rules

| Category | When to Use | Example |
|----------|-------------|---------||
| [Cat 1] | [Rule] | [Example] |
| [Cat 2] | [Rule] | [Example] |
| ... | ... | ... |

---

### Urgency Decision Rules

| Level | When to Use | Example |
|-------|-------------|---------||
| high | [Rule] | [Example] |
| medium | [Rule] | [Example] |
| low | [Rule] | [Example] |

---

### Known Limitations

1. [Limitation 1]
2. [Limitation 2]
3. [Limitation 3]

---

### Version History

| Version | Date | Changes | Category Acc | Urgency Acc |
|---------|------|---------|--------------|-------------|
| 1.0 | [date] | Initial | [X]% | [X]% |
| 2.0 | [date] | [Changes] | [X]% | [X]% |

---

### Production Deployment Notes

- **Human review required when:** [conditions]
- **Batch size recommendation:** [X] items per API call
- **Rate limiting:** [considerations]
- **Monitoring:** [what to track]

---

---

## Export and Submission

In [ ]:
# Export prompt log
if PROMPT_LOG:
    df_log = pd.DataFrame(PROMPT_LOG)
    df_log.to_csv("day2_assignment_prompt_log.csv", index=False)
    print("✓ Saved: day2_assignment_prompt_log.csv")
    print(df_log)

# Summary
print("\n" + "="*50)
print("SUBMISSION CHECKLIST")
print("="*50)
print(f"☐ Schema defined with Literal types")
print(f"☐ Final prompt with rules/examples")
print(f"☐ Input items: {len(inputs) if 'inputs' in dir() else 0} (need 12+)")
print(f"☐ Golden set: {len(GOLDEN) if 'GOLDEN' in dir() else 0} (need 8+)")
print(f"☐ Metrics computed")
print(f"☐ Error analysis written")
print(f"☐ Prompt playbook completed")
print("\nFiles to submit:")
print("  - This notebook (.ipynb or PDF)")
print("  - day2_assignment_extracted.json")
print("  - day2_assignment_prompt_log.csv")

---

**Congratulations on completing Day 2!**

You've learned how to:
- Use Pydantic schemas for guaranteed-valid structured outputs
- Build and iterate on extraction prompts
- Evaluate against golden sets
- Document production-ready prompt pipelines

Tomorrow: **Retrieval-Augmented Generation (RAG)** — grounding LLM responses in your own data!